In [0]:
-- Fetch the qdp_organisations_all document data product created by 01. Quantexa Processing of Raw Data in Bronze

-- 01-1. Create a point in time run_timestamp for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS run_timestamp;

DECLARE run_timestamp TIMESTAMP DEFAULT current_timestamp();

SELECT run_timestamp;


-- 01-2. Create default start date for all data inserts

DROP TEMPORARY VARIABLE IF EXISTS default_start_date;

DECLARE default_start_date STRING;

SET  VAR default_start_date = '01-01-1900';

SELECT default_start_date;


-- 01-3. Create default end date for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS default_end_date;

DECLARE default_end_date STRING;

SET  VAR default_end_date = '31-12-2999';

SELECT default_end_date;


 -- 2. Remove all Organisation records for the Tennant
DELETE FROM demo_banking_silver.qdp_links_all.link_organisation_address_history;


-- sat organisation
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_business_activity s
WHERE s.organisation_id IN (
  SELECT DISTINCT organisation_id
  FROM demo_banking_silver.qdp_organisations_all.hub_organisation
  WHERE tennant_id = :p_tennant_id)
;

-- sat communication method
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_communication_method s
WHERE s.organisation_id IN (
  SELECT DISTINCT organisation_id
  FROM demo_banking_silver.qdp_organisations_all.hub_organisation
  WHERE tennant_id = :p_tennant_id)
;

-- sat communication language
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_communication_language s
WHERE s.organisation_id IN (
  SELECT DISTINCT organisation_id
  FROM demo_banking_silver.qdp_organisations_all.hub_organisation
  WHERE tennant_id = :p_tennant_id)
;

-- sat industry sector
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_industry_sector s
WHERE s.organisation_id IN (
  SELECT DISTINCT organisation_id
  FROM demo_banking_silver.qdp_organisations_all.hub_organisation
  WHERE tennant_id = :p_tennant_id)
;

-- sat it system
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_it_system s
WHERE s.organisation_id IN (
  SELECT DISTINCT organisation_id
  FROM demo_banking_silver.qdp_organisations_all.hub_organisation
  WHERE tennant_id = :p_tennant_id)
;

-- sat industry type
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_organisation_identifier s
WHERE s.organisation_id IN (
  SELECT DISTINCT organisation_id
  FROM demo_banking_silver.qdp_organisations_all.hub_organisation
  WHERE tennant_id = :p_tennant_id)
;

-- sat organisation merged from
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_organisation_merged_from s
WHERE s.organisation_id IN (
  SELECT DISTINCT organisation_id
  FROM demo_banking_silver.qdp_organisations_all.hub_organisation
  WHERE tennant_id = :p_tennant_id)
;

-- sat organisation name history
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_organisation_name_history s
WHERE s.organisation_id IN (
  SELECT DISTINCT organisation_id
  FROM demo_banking_silver.qdp_organisations_all.hub_organisation
  WHERE tennant_id = :p_tennant_id)
;

-- sat organisation relationship
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_organisation_relationship s
WHERE s.organisation_id IN (
  SELECT DISTINCT organisation_id
  FROM demo_banking_silver.qdp_organisations_all.hub_organisation
  WHERE tennant_id = :p_tennant_id)
;

-- sat individual with significant control
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_individual_with_significant_control s
WHERE s.organisation_id IN (
  SELECT DISTINCT organisation_id
  FROM demo_banking_silver.qdp_organisations_all.hub_organisation
  WHERE tennant_id = :p_tennant_id)
;

-- sat organisation with significant control
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_organisation_with_significant_control s
WHERE s.organisation_id IN (
  SELECT DISTINCT organisation_id
  FROM demo_banking_silver.qdp_organisations_all.hub_organisation
  WHERE tennant_id = :p_tennant_id)
;

-- sat residence country
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_residence_country s
WHERE s.organisation_id IN (
  SELECT DISTINCT organisation_id
  FROM demo_banking_silver.qdp_organisations_all.hub_organisation
  WHERE tennant_id = :p_tennant_id)
;

-- sat organisation
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_organisation s
WHERE s.organisation_id IN (
  SELECT DISTINCT organisation_id
  FROM demo_banking_silver.qdp_organisations_all.hub_organisation
  WHERE tennant_id = :p_tennant_id)
;

-- Hub Organisation
DELETE FROM demo_banking_silver.qdp_organisations_all.hub_organisation WHERE tennant_id = :p_tennant_id;



-- 3. Create the table of organisation_entity_id and associated Addresses
CREATE OR REPLACE TEMP VIEW organisation_addresses AS
SELECT 
    organisation_entity_id, 
    address.address.address_entity_id,
    address.address.rank,
    address.address.is_primary_address,
    address.address.use_code,
    try_cast(address.address.use_concept_id AS BIGINT) AS use_concept_id,
    address.address.use_raw_code,
    try_cast(address.address.use_raw_concept_id AS BIGINT) AS use_raw_concept_id,
    try_to_timestamp(address.address.period_start, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(address.address.period_end, 'dd-MM-yyyy') AS period_end
FROM (
  SELECT org.organisation_entity_id, explode(address_history) AS address 
  FROM demo_banking_silver.qdp_from_quantexa_banking_supply_chain.qdp_organisations_all org
) AS exploded_addresses
;

SELECT * FROM organisation_addresses;

-- 4. Create the table of individual_entity_id and associated Resident Countries
CREATE OR REPLACE TEMP VIEW organisation_residence_countries AS
SELECT 
    organisation_entity_id, 
    resc.resident_country.country,
    resc.resident_country.country_code,
    try_cast(resc.resident_country.country_concept_id AS BIGINT) AS country_concept_id,
    resc.resident_country.country_raw_code,
    try_cast(resc.resident_country.country_raw_concept_id AS BIGINT) AS country_raw_concept_id,
    try_to_timestamp(resc.resident_country.period_start, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(resc.resident_country.period_end, 'dd-MM-yyyy') AS period_end
FROM (
  SELECT org.organisation_entity_id, explode(resident_countries) AS resc 
  FROM demo_banking_silver.qdp_from_quantexa_banking_supply_chain.qdp_organisations_all org
) AS exploded_resident_countries
;

SELECT * FROM organisation_residence_countries;


-- 5. Create the table of organisation_entity_id and associated Communication Methods
CREATE OR REPLACE TEMP VIEW organisation_communication_methods AS
SELECT 
    organisation_entity_id, 
    communication_method.communication_method.rank,
    communication_method.communication_method.is_primary_communication_method,
    communication_method.communication_method.value,
    communication_method.communication_method.value_display,
    communication_method.communication_method.value_raw,
    communication_method.communication_method.type_code,
    try_cast(communication_method.communication_method.type_concept_id AS BIGINT) AS type_concept_id,
    communication_method.communication_method.type_raw_code,
    try_cast(communication_method.communication_method.type_raw_concept_id AS BIGINT) AS type_raw_concept_id,
    communication_method.communication_method.use_code,
    try_cast(communication_method.communication_method.use_concept_id AS BIGINT) AS use_concept_id,
    communication_method.communication_method.use_raw_code,
    try_cast(communication_method.communication_method.use_raw_concept_id AS BIGINT) AS use_raw_concept_id,
    communication_method.communication_method.telephony_country_code,
    try_cast(communication_method.communication_method.telephony_country_concept_id AS BIGINT) AS telephony_country_concept_id,
    communication_method.communication_method.telephony_country_raw_code,
    try_cast(communication_method.communication_method.telephony_country_raw_concept_id AS BIGINT) AS telephony_country_raw_concept_id,
    try_to_timestamp(communication_method.communication_method.period_start, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(communication_method.communication_method.period_end, 'dd-MM-yyyy') AS period_end
FROM (
  SELECT org.organisation_entity_id, explode(communication_methods) AS communication_method 
  FROM demo_banking_silver.qdp_from_quantexa_banking_supply_chain.qdp_organisations_all org
) AS exploded_communication_methods
LIMIT 1000;

SELECT * FROM organisation_communication_methods;

-- 6. Create the table of individual_entity_id and associated individual_with_significant_control
CREATE OR REPLACE TEMP VIEW individuals_with_significant_control AS
SELECT 
    organisation_entity_id, 
    iwsc.individual_with_significant_control.controlling_individual_entity_id,
    iwsc.individual_with_significant_control.type_code,
    try_cast(iwsc.individual_with_significant_control.type_concept_id AS BIGINT) AS type_concept_id,
    iwsc.individual_with_significant_control.type_raw_code,
    try_cast(iwsc.individual_with_significant_control.type_raw_concept_id AS BIGINT) AS type_raw_concept_id,
    iwsc.individual_with_significant_control.role_code,
    try_cast(iwsc.individual_with_significant_control.role_concept_id AS BIGINT) AS role_concept_id,
    iwsc.individual_with_significant_control.role_raw_code,
    try_cast(iwsc.individual_with_significant_control.role_raw_concept_id AS BIGINT) AS role_raw_concept_id,
    CASE 
     WHEN iwsc.individual_with_significant_control.individual_dob = '' THEN NULL
     ELSE to_date(iwsc.individual_with_significant_control.individual_dob, 'dd-MM-yyyy')
    END AS individual_dob,
    try_cast(iwsc.individual_with_significant_control.shareholding AS DECIMAL(10,2)) AS shareholding,
    iwsc.individual_with_significant_control.notes,
    try_to_timestamp(iwsc.individual_with_significant_control.period_start, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(iwsc.individual_with_significant_control.period_end, 'dd-MM-yyyy') AS period_end
FROM (
  SELECT org.organisation_entity_id, explode(individuals_with_significant_control) AS iwsc 
  FROM demo_banking_silver.qdp_from_quantexa_banking_supply_chain.qdp_organisations_all org
) AS exploded_individuals_with_significant_control
;

SELECT * FROM individuals_with_significant_control;

/**
-- 7. Create the table of organisation_entity_id and associated individual_with_significant_control
CREATE OR REPLACE TEMP VIEW organisations_with_significant_control AS
SELECT 
    organisation_entity_id, 
    owsc.organisation_with_significant_control.controlling_organisation_entity_id,
    owsc.organisation_with_significant_control.type_code,
    try_cast(owsc.organisation_with_significant_control.type_concept_id AS BIGINT) AS type_concept_id,
    owsc.organisation_with_significant_control.type_raw_code,
    try_cast(owsc.organisation_with_significant_control.type_raw_concept_id AS BIGINT) AS type_raw_concept_id,
    owsc.organisation_with_significant_control.role_code,
    try_cast(owsc.organisation_with_significant_control.role_concept_id AS BIGINT) AS role_concept_id,
    owsc.organisation_with_significant_control.role_raw_code,
    try_cast(owsc.organisation_with_significant_control.role_raw_concept_id AS BIGINT) AS role_raw_concept_id,
    try_cast(owsc.organisation_with_significant_control.shareholding AS DECIMAL(10,2)) AS shareholding,
    owsc.organisation_with_significant_control.notes,
    try_to_timestamp(owsc.organisation_with_significant_control.period_start, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(owsc.organisation_with_significant_control.period_end, 'dd-MM-yyyy') AS period_end
FROM (
  SELECT org.organisation_entity_id, explode(organisations_with_significant_control) AS owsc 
  FROM demo_banking_silver.qdp_from_quantexa_banking_supply_chain.qdp_organisations_all org
) AS exploded_organisations_with_significant_control
;

SELECT * FROM organisations_with_significant_control;
**/

-- 9. Create the  hub_organisation table records
INSERT INTO demo_banking_silver.qdp_organisations_all.hub_organisation
(organisation_entity_id, tennant_id, period_start, period_end)
SELECT organisation_entity_id, tennant_id, period_start, period_end
FROM demo_banking_silver.qdp_from_quantexa_banking_supply_chain.qdp_organisations_all org;

SELECT * FROM demo_banking_silver.qdp_organisations_all.hub_organisation;



-- 10. Create the  sat_organisation table records
INSERT INTO demo_banking_silver.qdp_organisations_all.sat_organisation
    ( 
    organisation_id,
    load_datetime,
    record_source_id,
	  data_source_code,
	  data_source_concept_id,
	  data_source_raw_code,
	  data_source_raw_concept_id,
	  type_code,
	  type_concept_id,
	  type_raw_code,
	  type_raw_concept_id,
	  status_code,
	  status_concept_id,
	  status_raw_code,
	  status_raw_concept_id,
		organisation_name,
		organisation_name_raw,
		organisation_name_parsed,
		trading_name,
		trading_name_raw,
		alternative_name,
		alternative_name_raw,
		duns_number,
		acronym,
		organisation_description,
		organisation_level,
		parent_organisation_id,
		primary_adddress_id,
		legal_structure_type_code,
		legal_structure_type_concept_id,
		legal_structure_type_raw_code,
		legal_structure_type_raw_concept_id,
		ring_fence_status,
		company_category_code,
		company_category_concept_id,
		company_category_raw_code,
		company_category_raw_concept_id,
		company_registration_name,
		company_registration_number,
		company_registration_date,
		company_registration_date_numeric,
		company_registration_place,
		company_registration_country_code,
		company_registration_country_concept_id,
		company_registration_country_raw_code,
		company_registration_country_raw_concept_id,
		primary_operation_country_code,
		primary_operation_country_concept_id,
		primary_operation_country_raw_code,
		primary_operation_country_raw_concept_id,
		secondary_operation_country_code,
		secondary_operation_country_concept_id,
		secondary_operation_country_raw_code,
		secondary_operation_country_raw_concept_id,
		company_franchise_type_code,
		company_franchise_type_concept_id,
		company_franchise_type_raw_code,
		company_franchise_type_raw_concept_id,
		vat_number,
		vat_registration_date,
		vat_registration_date_numeric,
		giin,
		bic,
		iban,
		legal_entity_identifier,
		firm_reference_number,
		unique_tax_payer_reference,
		capiq,
		risk_rating_code,
		risk_rating_concept_id,
		risk_rating_raw_code,
		risk_rating_raw_concept_id,
		period_start,
		period_end
    )
SELECT
  h.organisation_id AS organisation_id,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
  q.data_source_code,
  q.data_source_concept_id,
  q.data_source_raw_code,
  q.data_source_raw_concept_id,
  q.type_code,
  q.type_concept_id,
  q.type_raw_code,
  q.type_raw_concept_id,
  q.status_code,
  q.status_concept_id,
  q.status_raw_code,
  q.status_raw_concept_id,
  q.organisation_name,
  q.organisation_name_raw,
  q.organisation_name_parsed,
  q.trading_name,
  q.trading_name_raw,
  q.alternative_name,
  q.alternative_name_raw,
  q.duns_number,
  q.acronym,
  q.organisation_description,
  q.organisation_level,
  q.parent_organisation_id,
  q.primary_adddress_id,
  q.legal_structure_type_code,
  q.legal_structure_type_concept_id,
  q.legal_structure_type_raw_code,
  q.legal_structure_type_raw_concept_id,
  q.ring_fence_status,
  q.company_category_code,
  q.company_category_concept_id,
  q.company_category_raw_code,
  q.company_category_raw_concept_id,
  q.company_registration_name,
  q.company_registration_number,
  q.company_registration_date,
  q.company_registration_date_numeric,
  q.company_registration_place,
  q.company_registration_country_code,
  q.company_registration_country_concept_id,
  q.company_registration_country_raw_code,
  q.company_registration_country_raw_concept_id,
  q.primary_operation_country_code,
  q.primary_operation_country_concept_id,
  q.primary_operation_country_raw_code,
  q.primary_operation_country_raw_concept_id,
  q.secondary_operation_country_code,
  q.secondary_operation_country_concept_id,
  q.secondary_operation_country_raw_code,
  q.secondary_operation_country_raw_concept_id,
  q.company_franchise_type_code,
  q.company_franchise_type_concept_id,
  q.company_franchise_type_raw_code,
  q.company_franchise_type_raw_concept_id,
  q.vat_number,
  q.vat_registration_date,
  q.vat_registration_date_numeric,
  q.giin,
  q.bic,
  q.iban,
  q.legal_entity_identifier,
  q.firm_reference_number,
  q.unique_tax_payer_reference,
  q.capiq,
  q.risk_rating_code,
  q.risk_rating_concept_id,
  q.risk_rating_raw_code,
  q.risk_rating_raw_concept_id,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM demo_banking_silver.qdp_from_quantexa_banking_supply_chain.qdp_organisations_all q
JOIN demo_banking_silver.qdp_organisations_all.hub_organisation h
  ON h.organisation_entity_id = q.organisation_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_organisations_all.sat_organisation;


-- 11. Create the  sat_communication_method table records

INSERT INTO demo_banking_silver.qdp_organisations_all.sat_communication_method
    ( 
     organisation_id, 
     load_datetime, 
     record_source_id,
     rank,
     is_primary_communication_method,
     value,
     value_display,
     value_raw,
     type_code,
     type_concept_id,
     type_raw_code,
     type_raw_concept_id,
     use_code,
     use_concept_id,
     use_raw_code,
     use_raw_concept_id,
     telephony_country_code,
     telephony_country_concept_id,
     telephony_country_raw_code,
     telephony_country_raw_concept_id,
     period_start,
     period_end 
    )
SELECT
  h.organisation_id AS organisation_id,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
  q.rank AS rank,
  q.is_primary_communication_method AS is_primary_communication_method,
  q.value AS value,
  q.value_display AS value_display, 
  q.value_raw AS value_raw, 
  q.type_code AS type_code,
  q.type_concept_id AS type_concept_id,
  q.type_raw_code AS type_raw_code,
  q.type_raw_concept_id AS type_raw_concept_id,
  q.use_code AS use_code,
  q.use_concept_id AS use_concept_id,
  q.use_raw_code AS use_raw_code,
  q.use_raw_concept_id AS use_raw_concept_id,
  q.telephony_country_code AS telephony_country_code,
  q.telephony_country_concept_id AS telephony_country_concept_id,
  q.telephony_country_raw_code AS telephony_country_raw_code,
  q.telephony_country_raw_concept_id AS telephony_country_raw_concept_id,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM organisation_communication_methods q
JOIN demo_banking_silver.qdp_organisations_all.hub_organisation h
  ON h.organisation_entity_id = q.organisation_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_organisations_all.sat_communication_method;

-- 12. Create the sat_residence_country table records

INSERT INTO demo_banking_silver.qdp_organisations_all.sat_residence_country
    ( 
     organisation_id, 
     load_datetime, 
     record_source_id,
     country,
     country_code,
     country_concept_id,
     country_raw_code,
     country_raw_concept_id,
     period_start,
     period_end 
    )
SELECT
  h.organisation_id AS organisation_id,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
  q.country AS country,
  q.country_code AS country_code,
  q.country_concept_id AS country_concept_id,
  q.country_raw_code AS country_raw_code,
  q.country_raw_concept_id AS country_raw_concept_id,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM organisation_residence_countries q
JOIN demo_banking_silver.qdp_organisations_all.hub_organisation h
  ON h.organisation_entity_id = q.organisation_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_organisations_all.sat_residence_country;


-- 13. Create the  link_organisation_address_history table records    organisation_addresses
INSERT INTO demo_banking_silver.qdp_links_all.link_organisation_address_history
    ( 
     organisation_id, 
     organisation_entity_id,
     address_id,
     address_entity_id,
     is_primary_address,
     use_code,
     use_concept_id,
     use_raw_code,
     use_raw_concept_id,
     period_start,
     period_end 
    )
SELECT
  h.organisation_id AS organisation_id,
  q.organisation_entity_id AS organisation_entity_id,
  ha.address_id AS address_id,
  q.address_entity_id AS address_entity_id,
  q.is_primary_address AS is_primary_address,
  q.use_code AS use_code,
  q.use_concept_id AS use_concept_id,
  q.use_raw_code AS use_raw_code,
  q.use_raw_concept_id AS use_raw_concept_id,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM organisation_addresses q
JOIN demo_banking_silver.qdp_organisations_all.hub_organisation h
  ON h.organisation_entity_id = q.organisation_entity_id
JOIN demo_banking_silver.qdp_locations_all.hub_address ha
  ON ha.address_entity_id = q.address_entity_id;

SELECT * FROM demo_banking_silver.qdp_links_all.link_organisation_address_history;


-- 14. Create the sat_individual_with_significant_control table records
INSERT INTO demo_banking_silver.qdp_organisations_all.sat_individual_with_significant_control
    ( 
     organisation_id, 
     load_datetime, 
     record_source_id,
     controlling_individual_id,
     type_code,
     type_concept_id,
     type_raw_code,
     type_raw_concept_id,
     role_code,
     role_concept_id,
     role_raw_code,
     role_raw_concept_id,     
     individual_dob,
     shareholding,
     notes,
     period_start,
     period_end 
    )
SELECT
  h.organisation_id AS organisation_id,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
  controlind.individual_id AS controlling_individual_id,
  q.type_code AS type_code,
  q.type_concept_id AS type_concept_id,
  q.type_raw_code AS type_raw_code,
  q.type_raw_concept_id AS type_raw_concept_id,
  q.role_code AS type_code,
  q.role_concept_id AS type_concept_id,
  q.role_raw_code AS type_raw_code,
  q.role_raw_concept_id AS type_raw_concept_id,
  q.individual_dob AS individual_dob,
  q.shareholding AS shareholding,
  q.notes AS notes,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM individuals_with_significant_control q
JOIN demo_banking_silver.qdp_organisations_all.hub_organisation h
  ON h.organisation_entity_id = q.organisation_entity_id
JOIN demo_banking_silver.qdp_individuals_all.hub_individual controlind
  ON controlind.individual_entity_id = q.controlling_individual_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_organisations_all.sat_individual_with_significant_control;

/**

-- 15. Create the sat_organisation_with_significant_control table records

INSERT INTO demo_banking_silver.qdp_organisations_all.sat_organisation_with_significant_control
    ( 
     organisation_id, 
     load_datetime, 
     record_source_id,
     controlling_organisation_id,
     type_code,
     type_concept_id,
     type_raw_code,
     type_raw_concept_id,
     role_code,
     role_concept_id,
     role_raw_code,
     role_raw_concept_id,     
     shareholding,
     notes,
     period_start,
     period_end 
    )
SELECT
  h.organisation_id AS organisation_id,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
  controlorg.organisation_id AS controlling_organisation_id,
  q.type_code AS type_code,
  q.type_concept_id AS type_concept_id,
  q.type_raw_code AS type_raw_code,
  q.type_raw_concept_id AS type_raw_concept_id,
  q.role_code AS type_code,
  q.role_concept_id AS type_concept_id,
  q.role_raw_code AS type_raw_code,
  q.role_raw_concept_id AS type_raw_concept_id,
  q.shareholding AS shareholding,
  q.notes AS notes,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM organisations_with_significant_control q
JOIN demo_banking_silver.qdp_organisations_all.hub_organisation h
  ON h.organisation_entity_id = q.organisation_entity_id
JOIN demo_banking_silver.qdp_organisations_all.hub_organisation controlorg
  ON controlorg.organisation_entity_id = q.controlling_organisation_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_organisations_all.sat_organisation_with_significant_control;

**/

